
# Cart2Insights — Step 4: Data Quality Analysis

## Business Objective

Identify missing values

Check duplicates

Check incorrect data types

Identify inconsistent or invalid values

Identify potential outliers

Check primary-key uniqueness


Import Libraries

In [1]:
import pandas as pd
import numpy as np

Load Datasets

In [2]:
customers = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_customers_dataset.csv')
geolocation = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_order_items_dataset.csv')
order_payments = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_order_reviews_dataset.csv')
orders = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_orders_dataset.csv')
products = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_sellers_dataset.csv')
category_translation = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/product_category_name_translation.csv')

Dataset inventory

In [3]:
datasets = {
    'customers': customers,
    'geolocation': geolocation,
    'order_items': order_items,
    'order_payments': order_payments,
    'order_reviews': order_reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'category_translation': category_translation
}
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
geolocation: (1000163, 5)
order_items: (112650, 7)
order_payments: (103886, 5)
order_reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


Identify missing values

In [4]:
for name, df in datasets.items():
    missing_counts = df.isnull().sum()
    missing_only = missing_counts[missing_counts > 0]
    print(f"=== {name} (Total rows: {len(df)}) ===")
    if not missing_only.empty:
        for col, count in missing_only.items():
            percentage = (count / len(df)) * 100
            print(f"  {col}: {count} missing values ({percentage:.2f}%)")
    else:
        print("  No missing values found!")
    print()

=== customers (Total rows: 99441) ===
  No missing values found!

=== geolocation (Total rows: 1000163) ===
  No missing values found!

=== order_items (Total rows: 112650) ===
  No missing values found!

=== order_payments (Total rows: 103886) ===
  No missing values found!

=== order_reviews (Total rows: 99224) ===
  review_comment_title: 87656 missing values (88.34%)
  review_comment_message: 58247 missing values (58.70%)

=== orders (Total rows: 99441) ===
  order_approved_at: 160 missing values (0.16%)
  order_delivered_carrier_date: 1783 missing values (1.79%)
  order_delivered_customer_date: 2965 missing values (2.98%)

=== products (Total rows: 32951) ===
  product_category_name: 610 missing values (1.85%)
  product_name_lenght: 610 missing values (1.85%)
  product_description_lenght: 610 missing values (1.85%)
  product_photos_qty: 610 missing values (1.85%)
  product_weight_g: 2 missing values (0.01%)
  product_length_cm: 2 missing values (0.01%)
  product_height_cm: 2 missin

### Check duplicates

In [5]:
for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    percentage = (duplicate_count / len(df)) * 100
    print(f"{name}: {duplicate_count} duplicate rows ({percentage:.2f}%)")

customers: 0 duplicate rows (0.00%)
geolocation: 261831 duplicate rows (26.18%)
order_items: 0 duplicate rows (0.00%)
order_payments: 0 duplicate rows (0.00%)
order_reviews: 0 duplicate rows (0.00%)
orders: 0 duplicate rows (0.00%)
products: 0 duplicate rows (0.00%)
sellers: 0 duplicate rows (0.00%)
category_translation: 0 duplicate rows (0.00%)


### Check incorrect data types

In [6]:
# Identify potential date columns and check their data types
date_related_datasets = ['orders', 'order_items', 'order_reviews']

for name in date_related_datasets:
    df = datasets[name]
    print(f"=== {name} dtypes ===")
    # Filter columns that contain 'date', 'time', or 'at' in their names
    date_cols = [col for col in df.columns if any(x in col for x in ['date', 'time', 'at', 'timestamp'])]
    for col in date_cols:
        print(f"  {col}: {df[col].dtype}")
    print()

=== orders dtypes ===
  order_status: object
  order_purchase_timestamp: object
  order_approved_at: object
  order_delivered_carrier_date: object
  order_delivered_customer_date: object
  order_estimated_delivery_date: object

=== order_items dtypes ===
  shipping_limit_date: object

=== order_reviews dtypes ===
  review_creation_date: object
  review_answer_timestamp: object



### Identify inconsistent or invalid values

In [7]:
# 1. Temporal consistency check
print("=== 1. Temporal Consistency Check (orders) ===")
orders_temp = orders.copy()
# Temporary conversion to datetimes for logical checking
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date']
for col in date_cols:
    orders_temp[col] = pd.to_datetime(orders_temp[col], errors='coerce')

# Check if approval happens before purchase
inconsistent_approved = orders_temp[orders_temp['order_approved_at'] < orders_temp['order_purchase_timestamp']]
print(f"Orders approved before purchase: {len(inconsistent_approved)}")

# Check if delivery to carrier happens before purchase
inconsistent_carrier = orders_temp[orders_temp['order_delivered_carrier_date'] < orders_temp['order_purchase_timestamp']]
print(f"Orders delivered to carrier before purchase: {len(inconsistent_carrier)}")

# Check if customer delivery happens before purchase
inconsistent_delivery = orders_temp[orders_temp['order_delivered_customer_date'] < orders_temp['order_purchase_timestamp']]
print(f"Orders delivered to customer before purchase: {len(inconsistent_delivery)}")

=== 1. Temporal Consistency Check (orders) ===
Orders approved before purchase: 0
Orders delivered to carrier before purchase: 166
Orders delivered to customer before purchase: 0


In [8]:
# 2. Numeric Range Consistency check
print("=== 2. Numeric Range Check ===")
invalid_prices = order_items[order_items['price'] <= 0]
print(f"Order items with price <= 0: {len(invalid_prices)}")

invalid_freight = order_items[order_items['freight_value'] < 0]
print(f"Order items with freight_value < 0: {len(invalid_freight)}")

invalid_payments = order_payments[order_payments['payment_value'] <= 0]
print(f"Payments with payment_value <= 0: {len(invalid_payments)}")

=== 2. Numeric Range Check ===
Order items with price <= 0: 0
Order items with freight_value < 0: 0
Payments with payment_value <= 0: 9


In [9]:
# 3. Geolocation bounds validation
print("=== 3. Geolocation Latitude/Longitude Range Check ===")
invalid_lat = geolocation[(geolocation['geolocation_lat'] < -90) | (geolocation['geolocation_lat'] > 90)]
invalid_lng = geolocation[(geolocation['geolocation_lng'] < -180) | (geolocation['geolocation_lng'] > 180)]
print(f"Invalid Latitudes (outside -90 to 90): {len(invalid_lat)}")
print(f"Invalid Longitudes (outside -180 to 180): {len(invalid_lng)}")

=== 3. Geolocation Latitude/Longitude Range Check ===
Invalid Latitudes (outside -90 to 90): 0
Invalid Longitudes (outside -180 to 180): 0


### Identify potential outliers

In [10]:
# Analyze numerical distributions to identify statistical outliers (using IQR method)
for df_name, col in [('order_items', 'price'), ('order_items', 'freight_value'), ('order_payments', 'payment_value')]:
    df = datasets[df_name]
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"=== Outliers in {df_name}.{col} ===")
    print(f"  IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"  Number of outliers: {len(outliers)} ({len(outliers)/len(df)*100:.2f}% of total)")
    print(f"  Max value: {df[col].max():.2f}")
    print()

=== Outliers in order_items.price ===
  IQR bounds: [-102.60, 277.40]
  Number of outliers: 8427 (7.48% of total)
  Max value: 6735.00

=== Outliers in order_items.freight_value ===
  IQR bounds: [0.98, 33.25]
  Number of outliers: 12134 (10.77% of total)
  Max value: 409.68

=== Outliers in order_payments.payment_value ===
  IQR bounds: [-115.78, 344.41]
  Number of outliers: 7981 (7.68% of total)
  Max value: 13664.08



### Check primary-key uniqueness

In [11]:
# Check uniqueness of primary keys
pks = {
    'customers': 'customer_id',
    'sellers': 'seller_id',
    'products': 'product_id',
    'orders': 'order_id'
}

for name, pk in pks.items():
    df = datasets[name]
    is_unique = df[pk].is_unique
    print(f"{name}.{pk} is unique: {is_unique} (Count: {df[pk].nunique()} vs Rows: {len(df)})")

customers.customer_id is unique: True (Count: 99441 vs Rows: 99441)
sellers.seller_id is unique: True (Count: 3095 vs Rows: 3095)
products.product_id is unique: True (Count: 32951 vs Rows: 32951)
orders.order_id is unique: True (Count: 99441 vs Rows: 99441)


### Identify potential outliers

In [12]:
# Analyze numerical distributions to identify statistical outliers (using IQR method)
for df_name, col in [('order_items', 'price'), ('order_items', 'freight_value'), ('order_payments', 'payment_value')]:
    df = datasets[df_name]
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"=== Outliers in {df_name}.{col} ===")
    print(f"  IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"  Number of outliers: {len(outliers)} ({len(outliers)/len(df)*100:.2f}% of total)")
    print(f"  Max value: {df[col].max():.2f}")
    print()

=== Outliers in order_items.price ===
  IQR bounds: [-102.60, 277.40]
  Number of outliers: 8427 (7.48% of total)
  Max value: 6735.00

=== Outliers in order_items.freight_value ===
  IQR bounds: [0.98, 33.25]
  Number of outliers: 12134 (10.77% of total)
  Max value: 409.68

=== Outliers in order_payments.payment_value ===
  IQR bounds: [-115.78, 344.41]
  Number of outliers: 7981 (7.68% of total)
  Max value: 13664.08



### Check primary-key uniqueness

In [13]:
# Check uniqueness of primary keys
pks = {
    'customers': 'customer_id',
    'sellers': 'seller_id',
    'products': 'product_id',
    'orders': 'order_id'
}

for name, pk in pks.items():
    df = datasets[name]
    is_unique = df[pk].is_unique
    print(f"{name}.{pk} is unique: {is_unique} (Count: {df[pk].nunique()} vs Rows: {len(df)})")

customers.customer_id is unique: True (Count: 99441 vs Rows: 99441)
sellers.seller_id is unique: True (Count: 3095 vs Rows: 3095)
products.product_id is unique: True (Count: 32951 vs Rows: 32951)
orders.order_id is unique: True (Count: 99441 vs Rows: 99441)
